Here we do an initial combination of the Open Air Chicago clarity sensors and satellite retrieved NO2 tropospheric column measurements.
Hourly and spatial aggregations on the open air data are performed. Basic statistic metrics are calculated. 
This notebook does not currently filter out pixels with single sites, but that is planned as a future addition per request of Carl Malings. 

- Authorship: Noah Ring
- Year: 2026
- Program: EarthRISE Developers Academy
- Location: Langley Research Center

In [1]:
import xarray as xr
import matplotlib as plt
import pandas as pd
import geopandas as gpd

# Change the matplotlib dpi to increase or decrease output size in vscode
plt.rcParams['figure.dpi'] = 150 

# Change some of the default xarray settings
xr.set_options(keep_attrs=True, display_expand_data=False)

# EPSG to be used when setting coordinate reference systems
epsg = 4326 

# Function used for coercing clarity lat/lon to match the 0.2 x 0.2deg grid of TEMPO
# Note: I included the bias term (0.01) to force the lat/lon to match TEMPO lat/lon
def round_002(x):
    out = 0.02*round(x/0.02) + 0.01
    return out


In [2]:

#Define the path where the tempo.nc file is stored.
tempo_path = "~/Documents/01_Research/larc-chicago-haq/datasets/tempo.nc"

# Define the path where the open air chicago clarity data is stored
clarity_path = "~/Documents/01_Research/larc-chicago-haq/datasets/Open_Air_Chicago_Individual_Measurements_20260615.csv"

# Define path where the shapefiles for chicago are stored.
# Note: This is currently unused in this file.
community_shapefiles = (
                        "~/Documents/01_Research/larc-chicago-haq/shapefiles/"
                        "ChicagoCommunityAreasBoundaries/geo_export_6383d95e-1e67-4fe2-8f45-534d73a53a92.shp"
                        )

# Use xarray to load tempo data
tempo_raster = xr.open_dataset(tempo_path, decode_coords=True, decode_times=True)
tempo_raster = tempo_raster.rename({"vertical_column_troposphere":"column_no2"})
tempo_raster

# Load in community boundaries
chi_communities = gpd.read_file(community_shapefiles)
# chi_communities = chi_communities.set_crs(epsg="EPSG:4326")

# Load clarity data .csv
clarity_full_dat = pd.read_csv(clarity_path)

# Subset the clarity sensor data for our variables of interest
# Reformat the datetime variable, and add other time columns to possibly be used later
clarity_subset = clarity_full_dat[['datasourceid', 'sensor_name', 'latitude', 'longitude', 'time',
                            'no2ConcIndividual.value']]
clarity_subset['datetime'] = pd.to_datetime(clarity_subset["time"], format="%m/%d/%Y %I:%M:%S %p")
clarity_subset['month'] = clarity_subset['datetime'].dt.month
clarity_subset["year"] = clarity_subset['datetime'].dt.year
clarity_subset["month_year_str"] = clarity_subset['month'].astype(str) + "-" + clarity_subset['year'].astype(str)
clarity_subset["month_year"] = pd.to_datetime(clarity_subset["month_year_str"], format="%m-%Y")

# Convert clarity subset to geopandas dataframe
clarity_subset["geometry"] = gpd.points_from_xy(
    x = clarity_subset["longitude"],
    y = clarity_subset["latitude"]
)

# Set coordinate reference system 
clarity_subset = gpd.GeoDataFrame(clarity_subset).set_crs(epsg=epsg)

# Apply rounding function to the lat/lon columns of our subset clarity data
clarity_subset['latitude'] = clarity_subset['latitude'].apply(lambda x: round_002(x))
clarity_subset['longitude'] = clarity_subset['longitude'].apply(lambda x: round_002(x))

### Combining TEMPO and Clarity - Hourly Aggregation

In [5]:

# Group by time and lat/lon. This will put multiple stations at the same lat/lon coordinate, and aggregate their data together.
# The 'freq' option in pd.Grouper can be changed to 'h' for doing diel cycle avgs over the whole study period
clarity_subset_hour = (
    clarity_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='1h'), 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std'),
        sensor_count=('datasourceid', 'nunique')
        ).reset_index()
)

# Calculate the hourly vars of interest for tempo per pixel
tempo_hour_mean = tempo_raster.column_no2.resample(time='1h').mean(dim='time')
tempo_hour_median = tempo_raster.column_no2.resample(time='1h').median(dim='time')
tempo_hour_sd = tempo_raster.column_no2.resample(time='1h').std(dim='time')
tempo_hour_var = tempo_raster.column_no2.resample(time='1h').var(dim='time')

# Combine the above into a single dataframe.
tempo_hour_stats = xr.merge([
    tempo_hour_mean.rename('column_no2_mean'),
    tempo_hour_median.rename('column_no2_median'),
    tempo_hour_sd.rename('column_no2_sd'),
    tempo_hour_var.rename('column_no2_var')
])

# Rename the clarity datetime column to match the tempo time column
clarity_tempo_hour = clarity_subset_hour.rename(columns={'datetime': 'time'})
# Define the index as spatial and temporal prior to converting to xarray
clarity_tempo_hour = clarity_tempo_hour.set_index(['time', 'latitude', 'longitude']).to_xarray()

# Normalize coord precision before merging so matching is consistent.
# This step is kind of hacky but necessary to ensure coordinates match exactly between clarity and tempo
# If there is a better way to do this, then please replace this. Without this step you get NaN values when combining.
clarity_tempo_hour = clarity_tempo_hour.assign_coords(
    latitude=((clarity_tempo_hour.latitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
    longitude=((clarity_tempo_hour.longitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
)

tempo_hour_stats = tempo_hour_stats.assign_coords(
    latitude=((tempo_hour_stats.latitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
    longitude=((tempo_hour_stats.longitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
)

# Here, we right join the clarity dataset to the tempo dataset.
# This keeps all points in tempo, but only matching points from clarity 
clarity_tempo_hour = xr.merge([clarity_tempo_hour, tempo_hour_stats], join='right', compat='override')

clarity_tempo_hour

output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-hourly.nc"
clarity_tempo_hour.to_netcdf(output_path)


/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slic

### Combing TEMPO and Clarity - Diurnal

In [6]:
clarity_subset_diel = (
    clarity_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='h'), 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std'),
        sensor_count=('datasourceid', 'nunique')
        ).reset_index()
)

tempo_diel_mean = tempo_raster.column_no2.resample(time='1h').mean(dim='time')
tempo_diel_median = tempo_raster.column_no2.resample(time='1h').median(dim='time')
tempo_diel_sd = tempo_raster.column_no2.resample(time='1h').std(dim='time')
tempo_diel_var = tempo_raster.column_no2.resample(time='1h').var(dim='time')

# Combine the above into a single dataframe.
tempo_diel_stats = xr.merge([
    tempo_diel_mean.rename('column_no2_mean'),
    tempo_diel_median.rename('column_no2_median'),
    tempo_diel_sd.rename('column_no2_sd'),
    tempo_diel_var.rename('column_no2_var')
])

# Rename the clarity datetime column to match the tempo time column
clarity_tempo_diel = clarity_subset_hour.rename(columns={'datetime': 'time'})
# Define the index as spatial and temporal prior to converting to xarray
clarity_tempo_diel = clarity_tempo_diel.set_index(['time', 'latitude', 'longitude']).to_xarray()

# Normalize coord precision before merging so matching is consistent.
# This step is kind of hacky but necessary to ensure coordinates match exactly between clarity and tempo
# If there is a better way to do this, then please replace this. Without this step you get NaN values when combining.
clarity_tempo_diel = clarity_tempo_diel.assign_coords(
    latitude=((clarity_tempo_diel.latitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
    longitude=((clarity_tempo_diel.longitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
)

tempo_diel_stats = tempo_diel_stats.assign_coords(
    latitude=((tempo_diel_stats.latitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
    longitude=((tempo_diel_stats.longitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
)

# Here, we right join the clarity dataset to the tempo dataset.
# This keeps all points in tempo, but only matching points from clarity 
clarity_tempo_diel = xr.merge([clarity_tempo_diel, tempo_diel_stats], join='right', compat='override')

clarity_tempo_diel

output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-diurnal.nc"
clarity_tempo_diel.to_netcdf(output_path)



/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slic

# TESTING BELOW HERE

Important: 
- Before we can aggregate clarity sensor data over days / weeks, we need to filter out nightime values consistent with TEMPO


In [ ]:


clarity_subset_daily = (
    clarity_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='1D'), 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std'),
        sensor_count=('datasourceid', 'nunique')
        ).reset_index()
)


clarity_subset_all_days = (
    clarity_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='1D'), 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std'),
        sensor_count=('datasourceid', 'nunique')
        ).reset_index()
)